# Watching PEST++ work, live

`solve()` blocks until its model runs are done, which on a real problem is all of the time.
This notebook makes that visible: a deliberately slow model, and a progress bar that moves
while the library is inside a single blocking call.

The model here is the ordinary 10-parameter cross-section case with a **sleep in front of the
forward run**. Nothing is faked -- the real model still runs and the results are real -- it is
just slow enough to watch. Change `SECONDS_PER_RUN` and re-run to taste.

> **This is the `with`-free version of `pestpp_progress_demo.ipynb`.** Same code, same results - the only
> difference is that sessions are held in a plain variable instead of a context manager.
>
> `with` was never required: `__enter__` just returns the session, and nothing checks
> whether you entered a block. It only ever called `close()` for you. This style is the
> better fit for a notebook, where a session has to outlive the cell that made it so you
> can poke at it from the next one.
>
> Cleanup still happens. `close()` is called explicitly below, and a session that is
> dropped without one is released by its finalizer - including the panther agents it
> started. That matters most when you re-run a setup cell: rebinding the name drops the
> old session, and without the finalizer its workers would go on running unnoticed.
>
> One `with` deliberately stays: `ies.par_view()`. That block is not bookkeeping, it is
> the lifetime of a borrowed buffer, and using the array outside it raises rather than
> returning stale memory.

In [ ]:
import os
import shutil
import sys

import pandas as pd

sys.path.insert(0, os.path.join('..', 'python'))
from pestpp import Ies

SECONDS_PER_RUN = 0.4     # how slow to make each forward run
N_REALS         = 10

BENCH = os.path.join('..', 'benchmarks')
os.environ['PATH'] += os.pathsep + os.path.abspath(
    os.path.join(BENCH, 'test_bin',
                 'win' if os.name == 'nt' else
                 ('mac' if sys.platform == 'darwin' else 'linux')))

wd = os.path.abspath('nbflat_slow')
if os.path.exists(wd):
    shutil.rmtree(wd)
shutil.copytree(os.path.join(BENCH, 'ies_10par_xsec', 'template'), wd)

# the slow wrapper: sleep, then run the model the control file already asked for
with open(os.path.join(wd, 'slow_model.py'), 'w') as f:
    f.write(
        'import subprocess, sys, time\n'
        'time.sleep({0})\n'.format(SECONDS_PER_RUN) +
        "subprocess.run(['mfnwt', '10par_xsec.nam'], check=False)\n")

import pyemu
pst = pyemu.Pst(os.path.join(wd, 'pest.pst'))
pst.model_command = ['python slow_model.py']
pst.control_data.noptmax = 2
pst.pestpp_options['ies_num_reals'] = N_REALS
pst.pestpp_options['random_seed'] = 11
pst.write(os.path.join(wd, 'pest.pst'), version=2)
print('each forward run will take about {0}s'.format(SECONDS_PER_RUN))

## The bar during a normal solve

Nothing about the loop below is special -- it is the ordinary `iterations()` loop. The only
difference is `progress=True`, and what it shows is **model runs**, counted from inside a
`solve()` that has not returned yet.

Watch the count reset as each iteration starts its own batch: ies runs a subset of the
ensemble against every lambda to pick a winner, then runs the rest at that winner, so one
iteration is two batches.

In [ ]:
ies = Ies.from_pst('pest.pst', workdir=wd)
ies.initialize()
for step in ies.iterations(progress=True):
    print('iteration {0}: phi {1:.4g}'.format(step.iter, step.phi_mean))
ies.finalize()

ies.close()

## The data behind the bar

The bar is a renderer over an *observer* the library calls while a batch is in flight. You can
register your own and do anything you like with the counters -- log them, plot them, decide
something.

Two things worth noticing in the output below:

- the observer fires during a plain `solve()`, which is the call that used to be completely
  silent
- it is **throttled inside the library**. Ask for `min_interval_sec=0.0` on this case and you
  get roughly 300,000 calls for a handful of runs, because the run manager's poll loop is
  hot. The throttle drops periodic ticks but never a run that finished, failed or timed out.

In [ ]:
wd2 = os.path.abspath('nbflat_slow_observer')
if os.path.exists(wd2):
    shutil.rmtree(wd2)
shutil.copytree(wd, wd2)

trace = []
ies = Ies.from_pst('pest.pst', workdir=wd2)
ies.initialize()
ies._lib.set_run_observer(
    lambda p: trace.append(dict(p)) or True,   # True = carry on
    min_interval_sec=0.25)
ies.solve()
ies._lib.set_run_observer(None)
ies.finalize()

ies.close()

df = pd.DataFrame(trace)
print('{0} notifications during one solve()'.format(len(df)))
df.tail(8)

## Stopping a batch from inside

The observer returns an action, not nothing. Returning `False` asks the run manager to stop
scheduling and bring the batch to an orderly end -- the runs already finished are kept.

This is the shape preemption will use. The plan in `docs/api_part1/panther_preemption.md` is
to ask workers for partial results from runs that are still going, look at what comes back,
and kill the ones not worth finishing. That is a new **return value** and new **fields**, not
a new callback: the observer already returns an action, and the struct it is handed carries a
size so it can grow.

The other half of that groundwork is what you are allowed to call from inside the observer.
It fires mid-batch, so the ensembles and phi are part-updated and reading them is meaningless
-- those calls are refused, by name. The run-management calls are not: `get_run_states`,
`get_run_time_stats`, `cancel_runs`, `get_worker_*`. That allowlist is exactly the door
preemption needs -- look at what is running, decide, cancel.

In [ ]:
wd3 = os.path.abspath('nbflat_slow_stop')
if os.path.exists(wd3):
    shutil.rmtree(wd3)
shutil.copytree(wd, wd3)

ies = Ies.from_pst('pest.pst', workdir=wd3)
ies.initialize(defer_runs=True)
queued = ies.queue_runs()

stopped_at = []

def stop_early(p):
    stopped_at.append(p['n_completed'])
    return p['n_completed'] < 4        # False stops the batch

ies._lib.set_run_observer(stop_early, min_interval_sec=0.0)
ies.run()
ies._lib.set_run_observer(None)
print('{0} runs queued, stopped after {1}'.format(queued, max(stopped_at)))
ies.close()

ies.close()

## Notes

**It writes to stderr, not stdout.** A session opened with `quiet=True` (the default) has the
library redirect file descriptor 1 to `pestpp.stdout.log`, and python's stdout is that same
descriptor -- so a bar printed to stdout would end up in the log file rather than on screen.
In a notebook the question does not arise: the display protocol is a kernel message.

**The observer runs on the calling thread**, never a worker thread. That is a rule the library
holds itself to rather than an accident: taking python's GIL from a thread it has never seen,
while the thread that called in sits blocked, is a deadlock rather than an error.

**An observer cannot fail a batch.** One that raises is unregistered rather than being allowed
to unwind through runs that are in flight, and an action the library does not recognise is
treated as "carry on".